In [1]:
!pip install -q \
    transformers \
    datasets \
    peft \
    trl \
    accelerate \
    bitsandbytes


In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer,BitsAndBytesConfig
from datasets import load_dataset


/home/anshagrawal/Desktop/launchpad/week8/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
dataset = load_dataset(
    "json",
    data_files={
        "train": "../data/train.jsonl",
        "validation": "../data/val.jsonl"
    }
)

dataset


Generating train split: 1816 examples [00:00, 238294.83 examples/s]
Generating validation split: 202 examples [00:00, 78398.21 examples/s]


DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 1816
    })
    validation: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 202
    })
})

In [8]:
from transformers import BitsAndBytesConfig
model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    trust_remote_code=True
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.float16
)


In [ ]:
from peft import prepare_model_for_kbit_training

model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)


In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


trainable params: 2,252,800 || all params: 1,102,301,184 || trainable%: 0.2044


In [9]:
def format_prompt(example):
    prompt = f"""### Instruction:
{example['instruction']}

### Input:
{example['input']}

### Response:
{example['output']}"""
    return {"text": prompt}

dataset = dataset.map(format_prompt)


Map: 100%|██████████| 202/202 [00:00<00:00, 15973.78 examples/s]


In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments


In [ ]:
training_args = TrainingArguments(
    output_dir="./lora-output",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=3,
    logging_steps=10,
    save_strategy="epoch",
    evaluation_strategy="epoch",
    fp16=True,
    optim="paged_adamw_8bit",
    report_to="none"
)


TypeError: TrainingArguments.__init__() got an unexpected keyword argument 'evaluation_strategy'

In [ ]:
training_args = TrainingArguments(
    output_dir="./lora-output",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=3,
    logging_steps=10,
    save_strategy="epoch",
    eval_strategy="epoch",
    bf16=True,
    fp16=False,
    optim="paged_adamw_8bit",
    report_to="none"
)


In [ ]:
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    tokenizer=tokenizer,
    args=training_args,
    dataset_text_field="text"
)

trainer.train()


TypeError: SFTTrainer.__init__() got an unexpected keyword argument 'tokenizer'

In [ ]:
from trl import SFTTrainer

def formatting_func(example):
    return example["text"]

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    processing_class=tokenizer,
    args=training_args,
    formatting_func=formatting_func
)

trainer.train()


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
1,0.357700,1.246008,0.577198,133516.000000,0.740044
2,0.264600,1.345269,0.515693,267032.000000,0.738137
3,0.236300,1.432813,0.483191,400548.000000,0.733481


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


TrainOutput(global_step=342, training_loss=0.41642747531857405, metrics={'train_runtime': 451.1096, 'train_samples_per_second': 12.077, 'train_steps_per_second': 0.758, 'total_flos': 3069314297806848.0, 'train_loss': 0.41642747531857405, 'epoch': 3.0})

In [ ]:
model.save_pretrained("adapters/finance-lora")
tokenizer.save_pretrained("adapters/finance-lora")


('adapters/finance-lora/tokenizer_config.json',
 'adapters/finance-lora/special_tokens_map.json',
 'adapters/finance-lora/chat_template.jinja',
 'adapters/finance-lora/tokenizer.model',
 'adapters/finance-lora/added_tokens.json',
 'adapters/finance-lora/tokenizer.json')

In [ ]:
from peft import PeftModel

base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    load_in_4bit=True,
    device_map="auto"
)

model = PeftModel.from_pretrained(base_model, "adapters/finance-lora")

prompt = """### Instruction:
Extract the total payable amount.

### Input:
Invoice 999: Consulting services $4,000, Tax $200, Total payable $4,200.

### Response:
"""

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False
)

print(tokenizer.decode(output[0], skip_special_tokens=True))


The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


### Instruction:
Extract the total payable amount.

### Input:
Invoice 999: Consulting services $4,000, Tax $200, Total payable $4,200.

### Response:
Payable amount: $3,800
